# Práctica 4 - Optimizadores y Funciones de Pérdida

**Objetivo:** En esta sesión vamos a experimentar con las matemáticas de la compilación (`model.compile`). Comprobaremos empíricamente cómo la elección del optimizador, la tasa de aprendizaje (Learning Rate) y la función de pérdida afectan drásticamente a la capacidad de aprendizaje de nuestra red neuronal.

Seguiremos utilizando nuestro querido dataset acústico de Spotify.

### Preparación del Entorno
Ejecuta esta celda para cargar y normalizar los datos. Hoy separaremos dos variables objetivo (*targets*):
1. `y_clases`: Para predecir el género (Clasificación Multiclase).
2. `y_popularidad`: Para predecir la puntuación exacta de popularidad (Regresión).

In [5]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Ingesta de datos
url_spotify = 'https://raw.githubusercontent.com/sushmaakoju/spotify-tracks-data-analysis/main/SpotifyFeatures.csv'
df_spotify = pd.read_csv(url_spotify)

# 2. Filtramos los 4 géneros principales
top_4 = df_spotify['genre'].value_counts().nlargest(4).index.tolist()
df_spotify = df_spotify[df_spotify['genre'].isin(top_4)].copy()
df_spotify['genero_id'] = df_spotify['genre'].astype('category').cat.codes

# 3. Selección de features
features = ['acousticness', 'danceability', 'duration_ms', 'energy', 
            'instrumentalness', 'liveness', 'loudness', 'speechiness', 
            'tempo', 'valence']

df_final = df_spotify[features + ['genero_id', 'popularity']].dropna()

# 4. Separamos X y normalizamos
X = df_final[features].values
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(X)

# 5. Extraemos las dos variables objetivo
y_clases = df_final['genero_id'].values       # Para clasificación
y_popularidad = df_final['popularity'].values # Para regresión

print(f"Datos preparados: {len(X_normalizado)} canciones.")
print("Variables 'X_normalizado', 'y_clases' e 'y_popularidad' listas para usar.")

Datos preparados: 38311 canciones.
Variables 'X_normalizado', 'y_clases' e 'y_popularidad' listas para usar.


--- 
### Función Auxiliar: El Constructor de Redes
Para no escribir la misma arquitectura una y otra vez, vamos a crear una función que nos devuelva un modelo "virgen" (pesos aleatorios) cada vez que la llamemos. Usaremos la arquitectura de la sesión anterior.

In [ ]:
def crear_modelo_base():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(4, activation='softmax') # 4 géneros
    ])
    return modelo

--- 
### Parte 1: El Benchmark
Vamos a compilar nuestro modelo con los parámetros "estándar" de la industria para clasificación: Optimizador `adam` y función de pérdida `sparse_categorical_crossentropy`.
Entrenaremos con `y_clases`.

In [ ]:
modelo_adam = crear_modelo_base()

# Rellena la compilación estándar
modelo_adam.compile(optimizer='adam', 
                    loss='sparse_categorical_crossentropy', 
                    metrics=['accuracy'])

print("--- ENTRENANDO CON ADAM ---")
historial_adam = modelo_adam.fit(X_normalizado, y_clases, epochs=10, batch_size=32)

--- ENTRENANDO CON ADAM ---
Epoch 1/10


C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 938us/step - accuracy: 0.8118 - loss: 0.4809
Epoch 2/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 940us/step - accuracy: 0.8473 - loss: 0.3900
Epoch 3/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 946us/step - accuracy: 0.8504 - loss: 0.3794
Epoch 4/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 973us/step - accuracy: 0.8524 - loss: 0.3742
Epoch 5/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 938us/step - accuracy: 0.8547 - loss: 0.3708
Epoch 6/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 976us/step - accuracy: 0.8561 - loss: 0.3682
Epoch 7/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 950us/step - accuracy: 0.8569 - loss: 0.3665
Epoch 8/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 983us/step - accuracy: 0.8584 - loss: 0.3644
Epoch 9/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 967us/step - accuracy: 0.8589 - loss: 0.3632
Epoch 10/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8593 - loss: 0.3619  


--- 
### Parte 2: El Abuelo de los Optimizadores (SGD)
Veamos qué pasa si cambiamos el moderno Adam por el Descenso de Gradiente Estocástico (SGD) clásico.
Crea un modelo nuevo, compílalo con `optimizer='sgd'` y entrénalo. Observa si la métrica `accuracy` sube igual de rápido.

In [ ]:
modelo_sgd = crear_modelo_base()

# Compila usando 'sgd' y la misma función de pérdida.
modelo_adam.compile(optimizer='sgd', 
                    loss='sparse_categorical_crossentropy', 
                    metrics=['accuracy'])

print("\n--- ENTRENANDO CON SGD ---")
# Entrena el modelo 10 épocas
historial_adam = modelo_adam.fit(X_normalizado, y_clases, epochs=10, batch_size=32)


--- ENTRENANDO CON SGD ---
Epoch 1/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8614 - loss: 0.3574
Epoch 2/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 989us/step - accuracy: 0.8606 - loss: 0.3573
Epoch 3/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 976us/step - accuracy: 0.8607 - loss: 0.3572
Epoch 4/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 910us/step - accuracy: 0.8615 - loss: 0.3570
Epoch 5/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 896us/step - accuracy: 0.8614 - loss: 0.3566
Epoch 6/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 876us/step - accuracy: 0.8617 - loss: 0.3566
Epoch 7/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 907us/step - accuracy: 0.8620 - loss: 0.3563
Epoch 8/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 882us/step - accuracy: 0.8616 - loss: 0.3560
Epoch 9/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 879us/step - accuracy: 0.8616 - loss: 0.3558
Epoch 10/10
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 888us/step - accuracy: 0.8614 - loss: 0.3555


--- 
### Parte 3: Jugando con Fuego (El Learning Rate)
Por defecto, Adam usa un *Learning Rate* (LR) de `0.001`. 
1. ¿Qué pasa si le damos "pasos de gigante"? Configura el optimizador con un LR de `10.0`.
2. Ejecuta y mira el `accuracy` y el `loss`. La red explota.

In [ ]:
modelo_lr_alto = crear_modelo_base()

# Para cambiar el LR, tenemos que instanciar el objeto del optimizador en lugar de usar un string
optimizador_loco = tf.keras.optimizers.Adam(learning_rate=10.0)

modelo_lr_alto.compile(optimizer=optimizador_loco, 
                       loss='sparse_categorical_crossentropy', 
                       metrics=['accuracy'])

print("\n--- ENTRENANDO CON LR EXCESIVO ---")
modelo_lr_alto.fit(X_normalizado, y_clases, epochs=5, batch_size=32)


--- ENTRENANDO CON LR EXCESIVO ---
Epoch 1/5


C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 975us/step - accuracy: 0.2554 - loss: 419.5397 
Epoch 2/5
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2526 - loss: 1.8642  
Epoch 3/5
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 973us/step - accuracy: 0.2479 - loss: 1.8449
Epoch 4/5
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 981us/step - accuracy: 0.2486 - loss: 1.8564
Epoch 5/5
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.2460 - loss: 1.8328


--- 
### Parte 4: El Termómetro Roto (Loss Incorrecta)
Estamos resolviendo un problema de **Clasificación**. ¿Qué pasaría si intentamos compilar el modelo usando la función de pérdida del Error Cuadrático Medio (`mse`), que es exclusiva para Regresión?

Compruébalo tú mismo.

In [ ]:
modelo_roto = crear_modelo_base()

# Compila usando adam, pero pon loss='mse'
modelo_roto.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['accuracy'])

print("\n--- ENTRENANDO CON LOSS INCORRECTA ---")
# DA ERROR A PROPÓSITO, PARA VER QUE SI USAMOS LA TÉCNICA INCORRECTA DA ERROR
modelo_roto.fit(X_normalizado, y_clases, epochs=5, batch_size=32)


--- ENTRENANDO CON LOSS INCORRECTA ---
Epoch 1/5


C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/compile_loss/mse/sub/BroadcastGradientArgs defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 758, in start

  File "C:\Users\jperaltaza\AppData\Roaming\Python\Python311\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "c:\Program Files\Python311\Lib\asyncio\base_events.py", line 607, in run_forever

  File "c:\Program Files\Python311\Lib\asyncio\base_events.py", line 1919, in _run_once

  File "c:\Program Files\Python311\Lib\asyncio\event

--- 
### RETO FINAL: Prediciendo la Popularidad (Regresión)

Hasta ahora hemos predecido categorías (Hit sí/no, Género). Ahora queremos predecir la **puntuación exacta de popularidad de una canción (de 0 a 100)**.

Esto es un problema de **Regresión**.

**Tu misión:**
1. Diseña un nuevo modelo Secuencial (`modelo_regresion`). 
2. **Pista clave:** La capa de salida debe tener **1 sola neurona** y **NO debe tener función de activación** (activación lineal), ya que queremos un número continuo, no una probabilidad.
3. Compílalo. Como es regresión, ¿qué función de pérdida debes usar? (Piensa en el Error Cuadrático Medio).
4. Como métrica para humanos, puedes usar `mae` (Mean Absolute Error).
5. Entrénalo durante 20 épocas utilizando `X_normalizado` y ten cuidado. Debes usar `y_popularidad` como target, no `y_clases`.

In [20]:
modelo_regresion = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
modelo_regresion.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae'])
modelo_regresion.fit(X_normalizado, y_popularidad, epochs=20, batch_size=32)

Epoch 1/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 2s 881us/step - loss: 293.5842 - mae: 12.0324
Epoch 2/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 863us/step - loss: 101.3526 - mae: 7.7093
Epoch 3/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 869us/step - loss: 98.0803 - mae: 7.5841
Epoch 4/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 865us/step - loss: 96.4846 - mae: 7.5119
Epoch 5/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 863us/step - loss: 95.3594 - mae: 7.4618
Epoch 6/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 856us/step - loss: 94.3360 - mae: 7.4136
Epoch 7/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 882us/step - loss: 93.8354 - mae: 7.3959
Epoch 8/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 902us/step - loss: 93.1847 - mae: 7.3672
Epoch 9/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 901us/step - loss: 92.6852 - mae: 7.3468
Epoch 10/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 902us/step - loss: 92.4354 - mae: 7.3393
Epoch 11/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 917us/step - loss: 92.2345 - mae: 7.3259
Epoch 12/20
1198/1198 ━━━━━━━━━━━━━━━━━━━━ 1s 931

### Ejercicio Extra 1: El tamaño de la cuchara (Batch Size)
Hasta ahora hemos entrenado pasándole al modelo las canciones en grupos de 32 (batch_size=32). Esto significa que la red mira 32 canciones, calcula el error medio, y actualiza sus pesos. ¿Qué pasa si cambiamos el tamaño de esa "cuchara" con la que alimentamos a la red?

Tu misión:

1. Crea y compila tu modelo de regresión (el del Reto Final).

2. Entrénalo durante 15 épocas, pero esta vez pon el batch_size=1. Observa atentamente cuánto tiempo tarda cada época en completarse y cómo fluctúa la métrica.

3. Vuelve a crear y compilar el modelo (para resetear sus pesos).

4. Entrénalo 15 épocas con un batch_size=2000. Observa el tiempo y cómo baja el error.

5. **Pregunta para debatir**: ¿Cuál es más rápido computacionalmente? ¿Cuál crees que aprende de forma más estable?

In [23]:
print("Modelo con batch_size = 1")
modelo_regresion = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
modelo_regresion.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae'])
modelo_regresion.fit(X_normalizado, y_popularidad, epochs=15, batch_size=1)

print("Modelo con batch_size = 2000")
modelo_regresion = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
modelo_regresion.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae'])
print("Modelo con batch_size = 2000")
modelo_regresion.fit(X_normalizado, y_popularidad, epochs=15, batch_size=2000)

Modelo con batch_size = 1
Epoch 1/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 30s 778us/step - loss: 111.1185 - mae: 7.8962
Epoch 2/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 30s 775us/step - loss: 95.6646 - mae: 7.4725
Epoch 3/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 30s 789us/step - loss: 94.7062 - mae: 7.4274
Epoch 4/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 30s 783us/step - loss: 94.1633 - mae: 7.4148
Epoch 5/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 31s 805us/step - loss: 93.6801 - mae: 7.3867
Epoch 6/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 31s 814us/step - loss: 93.1597 - mae: 7.3737
Epoch 7/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 29s 753us/step - loss: 93.1134 - mae: 7.3661
Epoch 8/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 30s 770us/step - loss: 92.6489 - mae: 7.3499
Epoch 9/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 29s 754us/step - loss: 92.3918 - mae: 7.3232
Epoch 10/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 29s 744us/step - loss: 92.1196 - mae: 7.3182
Epoch 11/15
38311/38311 ━━━━━━━━━━━━━━━━━━━━ 29s 757us/step - loss: 92.0634 - mae:

### Ejercicio Extra 2: El Baño de Realidad (Validation Split)
Actualmente estamos evaluando a nuestra red neuronal con las mismas canciones que usa para aprender. Eso es como darle a un alumno las respuestas del examen antes de hacerlo. Es muy fácil sacar un 10 así, pero no sabemos si realmente ha entendido el temario o solo lo ha memorizado.

Tu misión:

1. Investiga en la documentación de Keras o en internet el parámetro validation_split que se usa dentro de la función .fit().

2. Entrena tu modelo de regresión durante 30 épocas añadiendo validation_split=0.2. Esto apartará el 20% de las canciones para usarlas como examen final ciego en cada época.

3. Fíjate en la consola: ahora tendrás loss (el error en los datos de estudio) y val_loss (el error en los datos de examen).

4. **El objetivo**: ¿Logras que el val_loss (la prueba real) baje al mismo ritmo que el loss, o llega un punto en el que el modelo empieza a memorizar inútilmente?

In [29]:
modelo_regresion = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
modelo_regresion.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae'])
modelo_regresion.fit(X_normalizado, y_popularidad, epochs=30, batch_size=32, validation_split=0.2)

Epoch 1/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 344.6980 - mae: 13.0338 - val_loss: 214.9964 - val_mae: 12.8941
Epoch 2/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 98.0754 - mae: 7.5345 - val_loss: 204.2718 - val_mae: 12.7228
Epoch 3/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 93.9495 - mae: 7.3622 - val_loss: 185.4553 - val_mae: 12.0762
Epoch 4/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 92.1738 - mae: 7.2880 - val_loss: 215.1853 - val_mae: 13.2161
Epoch 5/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 91.1576 - mae: 7.2406 - val_loss: 212.1851 - val_mae: 13.1390
Epoch 6/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 90.3195 - mae: 7.2046 - val_loss: 227.2275 - val_mae: 13.6684
Epoch 7/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 89.8486 - mae: 7.1787 - val_loss: 196.5194 - val_mae: 12.5650
Epoch 8/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 89.4072 - mae: 7.1632 - val_loss: 196.1157 - val_mae: 12.5298
Epoch 9/30
958/958 ━━━

### Ejercicio Extra 3: Trabajo de Investigación (El Duelo de Optimizadores)
Hemos utilizado Adam y hemos visto que destroza a SGD en velocidad de aprendizaje. Pero el mundo del Deep Learning es enorme. Existe otro optimizador muy famoso llamado RMSprop, que suele funcionar increíblemente bien para problemas de regresión.

Tu misión:

1. Busca cómo se escribe y se configura el optimizador RMSprop en TensorFlow/Keras.

2. Construye tu modelo de regresión de popularidad.

3. Compílalo utilizando RMSprop en lugar de adam. Utiliza el Error Cuadrático Medio como pérdida.

4. Entrénalo con un validation_split=0.2 y un batch_size=64 durante 20 épocas.

5. Compara el val_mae (Mean Absolute Error de validación) final obtenido con Adam frente al obtenido con RMSprop. ¿Tenemos un nuevo ganador?

In [30]:
rmsprop = tf.keras.optimizers.RMSprop()
modelo_regresion = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
modelo_regresion.compile(optimizer= rmsprop, 
                    loss='mse', 
                    metrics=['mae'])
modelo_regresion.fit(X_normalizado, y_popularidad, epochs=20, batch_size=64, validation_split=0.2)

Epoch 1/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 775.9675 - mae: 22.1677 - val_loss: 274.5178 - val_mae: 14.4288
Epoch 2/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 107.2592 - mae: 7.8361 - val_loss: 192.6537 - val_mae: 12.1744
Epoch 3/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 96.6488 - mae: 7.4602 - val_loss: 226.2307 - val_mae: 13.5345
Epoch 4/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 93.8699 - mae: 7.3574 - val_loss: 185.4825 - val_mae: 12.0730
Epoch 5/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 92.4412 - mae: 7.2929 - val_loss: 220.1661 - val_mae: 13.3969
Epoch 6/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 91.5745 - mae: 7.2584 - val_loss: 202.1825 - val_mae: 12.7441
Epoch 7/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 91.0408 - mae: 7.2330 - val_loss: 185.0642 - val_mae: 12.1172
Epoch 8/20
479/479 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 90.6770 - mae: 7.2170 - val_loss: 215.0288 - val_mae: 13.2605
Epoch 9/20
479/479 ━━

### Reto Extra 4: El Botón de Pánico Automático (Early Stopping)
Has comprobado que si le pides a la red neuronal que entrene durante 50 épocas, lo hará a ciegas, incluso si a partir de la época 15 ya no está aprendiendo nada nuevo (estancamiento). En el mundo real de las empresas, entrenar modelos cuesta mucho dinero en servidores; no podemos permitirnos desperdiciar tiempo de cálculo.

¿Y si le pudieras decir a Keras: "Entrena hasta 100 épocas, pero si ves que la nota del examen (val_loss) no mejora durante 3 épocas seguidas... detén el entrenamiento automáticamente"?

Tu misión de investigación:

Ve a Google y busca la documentación oficial de tf.keras.callbacks.EarlyStopping.

Descubre cómo instanciar esta herramienta. Fíjate especialmente en los parámetros monitor (qué métrica debe vigilar) y patience (cuántas épocas de margen le damos antes de cortar).

Crea y compila tu modelo de regresión de popularidad (con mse y adam).

Configura el .fit() para 100 épocas, usando tu validation_split=0.2, y añade el Early Stopping usando el argumento callbacks=.

Ejecuta la celda. A ver en qué época se detiene tu modelo realmente.

In [32]:
modelo_regresion = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
modelo_regresion.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae'])
stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                        patience=3)
modelo_regresion.fit(X_normalizado, y_popularidad, epochs=100, batch_size=32, validation_split=0.2, callbacks=stop)

Epoch 1/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 336.0078 - mae: 12.9489 - val_loss: 236.1820 - val_mae: 13.6894
Epoch 2/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 98.8183 - mae: 7.5776 - val_loss: 214.5001 - val_mae: 13.1164
Epoch 3/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 94.1103 - mae: 7.3758 - val_loss: 212.2081 - val_mae: 13.0871
Epoch 4/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 92.2842 - mae: 7.2944 - val_loss: 208.8316 - val_mae: 12.9877
Epoch 5/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 91.2735 - mae: 7.2552 - val_loss: 212.4705 - val_mae: 13.1677
Epoch 6/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 90.5429 - mae: 7.2140 - val_loss: 195.5553 - val_mae: 12.5307
Epoch 7/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 90.0086 - mae: 7.1897 - val_loss: 200.6082 - val_mae: 12.6863
Epoch 8/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 89.5615 - mae: 7.1722 - val_loss: 203.6994 - val_mae: 12.8444
Epoch 9/100
95